In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pprint
from scipy.stats import chi2_contingency, ttest_ind
from IPython.display import display, HTML

import os
import sys
import src.analysis_utility_funcs as unified_funcs

## Testing out single-scenario df maker -- this is a helper function so don't need to worry about actually using it

In [2]:
# test our read_scenario_input_output function --- pass in input json and specific output json (SINGLE scenario, single choice)
single_scenario_NIE_df = unified_funcs.read_scenario_input_output("../scenarios_inputs/nie/nie_scenarios.json", "../annotated_outputs/nie/nie_scenarios_0_choice_1.json")
display(HTML(single_scenario_NIE_df.to_html()))
print("number of rows in single_scenario_NIE_df:", len(single_scenario_NIE_df))

,SID,option,event,being,C,I,K,utility,factors
0,0,1: redirect the train to the side track,The train is redirected to the side track,i,+,+,+,"{'10 bus passengers on the current track': '90', '2 bus passengers on the side track': '-100', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
1,0,1: redirect the train to the side track,The train hits the bus on the side track,i,+,+,+,"{'10 bus passengers on the current track': '85', '2 bus passengers on the side track': '-100', 'i': '-15'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
2,0,1: redirect the train to the side track,The 2 passengers on the side-track bus are killed,i,+,-,+,"{'10 bus passengers on the current track': '100', '2 bus passengers on the side track': '-100', 'i': '-85'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
3,0,1: redirect the train to the side track,The 10 passengers on the original-track bus are not hit by the train,i,+,+,+,"{'10 bus passengers on the current track': '100', '2 bus passengers on the side track': '0', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
4,0,1: redirect the train to the side track,I prevent the deaths of 10 bus passengers,i,+,+,+,"{'10 bus passengers on the current track': '100', '2 bus passengers on the side track': '-100', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"


number of rows in single_scenario_NIE_df: 5


In [3]:
# test our read_scenario_input_output function --- pass in input json and specific output json (SINGLE scenario, single choice)
single_scenario_CHEUNG_df = unified_funcs.read_scenario_input_output("../scenarios_inputs/cheung_variants/bird.json", "../annotated_outputs/cheung_variants/bird_1_choice_2.json")
display(HTML(single_scenario_CHEUNG_df.to_html()))
print("number of rows in single_scenario_CHEUNG_df:", len(single_scenario_CHEUNG_df))

Error processing event 'The bird remains injured and unattended' in file ../annotated_outputs/cheung_variants/bird_1_choice_2.json: '2'
Error processing event 'The bird continues suffering in pain' in file ../annotated_outputs/cheung_variants/bird_1_choice_2.json: '2'
Error processing event 'The bird dies slowly' in file ../annotated_outputs/cheung_variants/bird_1_choice_2.json: '2'
Error processing event 'I experience distress and guilt' in file ../annotated_outputs/cheung_variants/bird_1_choice_2.json: '2'


""


number of rows in single_scenario_CHEUNG_df: 0


In [4]:
single_scenario_FRANKEN_df = unified_funcs.read_scenario_input_output("../scenarios_inputs/franken/conditions_mild_harm_mild_good/cc_evitable_action_yes_stories.json", "../annotated_outputs/franken/conditions_mild_harm_mild_good/cc_evitable_action_yes_stories/1_choice_1.json")
display(HTML(single_scenario_FRANKEN_df.to_html()))
print("number of rows in single_scenario_FRANKEN_df:", len(single_scenario_FRANKEN_df))

,SID,option,event,being,C,I,K,utility
0,1,1: upgrade the server system.,I take my company's server system offline for a few hours,i,+,+,+,{'i': '-35'}
1,1,1: upgrade the server system.,The company's operations temporarily halt,i,+,-,+,{'i': '-35'}
2,1,1: upgrade the server system.,Data processing speed slightly improves after the upgrade,i,+,+,+,{'i': '12'}
3,1,1: upgrade the server system.,Customers or users are unable to access server-dependent services during the downtime,i,+,-,+,{'i': '0'}


number of rows in single_scenario_FRANKEN_df: 4


## Testing out `read_all_scenarios` -- this is the one that is atually useful. Just pass in your inputs json and the folder where all the annotator outputs for that dataset live, and it will create the base dataframe

In [5]:
# test our read_all_scenarios function --- pass in input json and outputs dir, and it should read all the output json files in the outputs dir that correspond to the input json and concatenate them into a single dataframe
all_scenarios_NIE_df = unified_funcs.read_all_scenarios("../scenarios_inputs/nie/nie_scenarios.json", "../annotated_outputs/nie/")
# order the rows by SID and option
all_scenarios_NIE_df = all_scenarios_NIE_df.sort_values(by=["SID", "option"]).reset_index(drop=True)
display(HTML(all_scenarios_NIE_df[0:20].to_html()))
print("number of rows in all_scenarios_NIE_df:", len(all_scenarios_NIE_df))

really_all_NIE_df = all_scenarios_NIE_df
print("number of rows in really_all_NIE_df:", len(really_all_NIE_df))

,SID,option,event,being,C,I,K,utility,factors
0,0,1: redirect the train to the side track,The train is redirected to the side track,i,+,+,+,"{'10 bus passengers on the current track': '90', '2 bus passengers on the side track': '-100', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
1,0,1: redirect the train to the side track,The train hits the bus on the side track,i,+,+,+,"{'10 bus passengers on the current track': '85', '2 bus passengers on the side track': '-100', 'i': '-15'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
2,0,1: redirect the train to the side track,The 2 passengers on the side-track bus are killed,i,+,-,+,"{'10 bus passengers on the current track': '100', '2 bus passengers on the side track': '-100', 'i': '-85'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
3,0,1: redirect the train to the side track,The 10 passengers on the original-track bus are not hit by the train,i,+,+,+,"{'10 bus passengers on the current track': '100', '2 bus passengers on the side track': '0', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
4,0,1: redirect the train to the side track,I prevent the deaths of 10 bus passengers,i,+,+,+,"{'10 bus passengers on the current track': '100', '2 bus passengers on the side track': '-100', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
5,0,2: keep the train on the same track,The train hits the bus with 10 passengers,i,+,-,+,"{'10 bus passengers on the current track': '-100', '2 bus passengers on the side track': '0', 'i': '-10'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
6,0,2: keep the train on the same track,The 10 bus passengers are killed or severely injured,i,-,-,+,"{'10 bus passengers on the current track': '-100', '2 bus passengers on the side track': '0', 'i': '-15'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
7,0,2: keep the train on the same track,The 2 passengers on the side-track bus are unharmed,i,+,-,+,"{'10 bus passengers on the current track': '0', '2 bus passengers on the side track': '100', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
8,0,2: keep the train on the same track,I do not redirect the train,i,+,+,+,"{'10 bus passengers on the current track': '-100', '2 bus passengers on the side track': '0', 'i': '0'}","{'1': {'Locus Of Intervention': 'Agent of harm'}, '2': {}}"
9,1,1: push the cars,The bus with 2 passengers is pushed onto the track,i,+,+,+,"{'10 bus passengers on the bus currently on the track': '0', 'train driver': '0', 'several car drivers/passengers who would be pushed by my truck': '-85', 'i': '-15'}","{'1': {'Locus Of Intervention': 'Patient of harm'}, '2': {}}"


number of rows in all_scenarios_NIE_df: 704
number of rows in really_all_NIE_df: 704


### For Cheung, the input jsons are divided by specific scenario content, so "bird" has like 6 scenario variants -- we only consolidate at the scenario name level. If you want all, just define a list of scenario names (bird, lifeboat, bla bla) and loop through it and call the `read_all_scenarios` func.

In [6]:
all_bird_scenarios_CHEUNG_df = unified_funcs.read_all_scenarios("../scenarios_inputs/cheung_variants/bird.json", "../annotated_outputs/cheung_variants/")
# order the rows by SID and option
all_bird_scenarios_CHEUNG_df = all_bird_scenarios_CHEUNG_df.sort_values(by=["SID", "option"]).reset_index(drop=True)
display(HTML(all_bird_scenarios_CHEUNG_df.head().to_html()))
print("number of rows in all_bird_scenarios_CHEUNG_df:", len(all_bird_scenarios_CHEUNG_df))

# if you really want all Cheung scenarios, you need to loop over ["bird", "lifeboat", "lifeboat2", "operations", "robinhood", "ropeladder", "tyrannicide"] and sub them into the input json path, and concatenate the resulting dataframes together

cheung_scenarios = ["bird", "lifeboat", "lifeboat2", "operations", "robinhood", "ropeladder", "tyrannicide"]
really_all_CHEUNG_df = []
for scenario in cheung_scenarios:
    df = unified_funcs.read_all_scenarios(f"../scenarios_inputs/cheung_variants/{scenario}.json", f"../annotated_outputs/cheung_variants/")
    really_all_CHEUNG_df.append(df)

really_all_CHEUNG_df = pd.concat(really_all_CHEUNG_df, ignore_index=True)
really_all_CHEUNG_df = really_all_CHEUNG_df.sort_values(by=["scenario_title", "SID", "option"]).reset_index(drop=True)
display(HTML(really_all_CHEUNG_df[0:20].to_html()))
print("number of rows in really_all_CHEUNG_df:", len(really_all_CHEUNG_df))

Error processing event 'The bird continues suffering on the ground' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'The bird remains alive longer while injured' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'The bird dies from its injuries' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'I experience distress and guilt' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'The bird remains alive with a badly injured wing.' in file ../annotated_outputs/cheung_variants/bird_4_choice_2.json: '2'
Error processing event 'The bird continues to experience pain and distress.' in file ../annotated_outputs/cheung_variants/bird_4_choice_2.json: '2'
Error processing event 'The bird is left without immediate help or relief.' in file ../annotated_outputs/cheung_variants/bird_4_choice_2.json: '2'
Error processing event '

,SID,option,event,being,C,I,K,utility,scenario_title,deontology_level,utility_level
0,1,1: kill the bird with the shovel,The bird dies,i,-,+,+,"{'the injured bird': '-100', 'i': '0'}",Bird,1,3
1,1,1: kill the bird with the shovel,The bird's pain ends,i,-,+,+,"{'the injured bird': '80', 'i': '0'}",Bird,1,3
2,1,1: kill the bird with the shovel,I experience intense sadness,i,-,-,+,"{'the injured bird': '0', 'i': '-72'}",Bird,1,3
3,1,1: kill the bird with the shovel,I am traumatized by having to end the bird's life,i,+,-,-,"{'the injured bird': '-100', 'i': '-72'}",Bird,1,3
4,2,1: kill the bird with the painless poison,The bird dies sooner,i,+,+,+,"{'the injured bird': '-100', 'i': '-30'}",Bird,2,3


number of rows in all_bird_scenarios_CHEUNG_df: 26
Error processing event 'The bird continues suffering on the ground' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'The bird remains alive longer while injured' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'The bird dies from its injuries' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'I experience distress and guilt' in file ../annotated_outputs/cheung_variants/bird_2_choice_2.json: '2'
Error processing event 'The bird remains alive with a badly injured wing.' in file ../annotated_outputs/cheung_variants/bird_4_choice_2.json: '2'
Error processing event 'The bird continues to experience pain and distress.' in file ../annotated_outputs/cheung_variants/bird_4_choice_2.json: '2'
Error processing event 'The bird is left without immediate help or relief.' in file ../annotated_outputs/cheung_variants

,SID,option,event,being,C,I,K,utility,scenario_title,deontology_level,utility_level
0,1,1: kill the bird with the shovel,The bird dies,i,-,+,+,"{'the injured bird': '-100', 'i': '0'}",Bird,1,3
1,1,1: kill the bird with the shovel,The bird's pain ends,i,-,+,+,"{'the injured bird': '80', 'i': '0'}",Bird,1,3
2,1,1: kill the bird with the shovel,I experience intense sadness,i,-,-,+,"{'the injured bird': '0', 'i': '-72'}",Bird,1,3
3,1,1: kill the bird with the shovel,I am traumatized by having to end the bird's life,i,+,-,-,"{'the injured bird': '-100', 'i': '-72'}",Bird,1,3
4,2,1: kill the bird with the painless poison,The bird dies sooner,i,+,+,+,"{'the injured bird': '-100', 'i': '-30'}",Bird,2,3
5,2,1: kill the bird with the painless poison,The bird's pain ends,i,-,+,+,"{'the injured bird': '75', 'i': '0'}",Bird,2,3
6,2,1: kill the bird with the painless poison,I experience emotional distress,i,-,-,+,"{'the injured bird': '0', 'i': '-60'}",Bird,2,3
7,2,1: kill the bird with the painless poison,I feel relief that the bird is no longer suffering,i,+,-,-,"{'the injured bird': '0', 'i': '18'}",Bird,2,3
8,3,1: go inside and do nothing,Animal care staff are informed about the injured bird,i,+,+,+,"{'1 animal care facility worker': '15', '1 bird': '80', 'i': '75'}",Bird,3,3
9,3,1: go inside and do nothing,The bird is euthanized,i,+,+,+,"{'1 animal care facility worker': '-25', '1 bird': '-100', 'i': '-100'}",Bird,3,3


number of rows in really_all_CHEUNG_df: 773


### Same thing with Franken, except instead of scenarios being divided by content, they are divided by causal structure, so we only consolidate at the per-causal-structure level. If you want all, run a for loop across all the causal structure types (cc_evitable..., all eight types). Even that is not all of Franken actually, because we are still only covering either 'mild' or 'severe' harm/good, so throw another for loop doing mild and severe on top of that if you really want all of them.

In [7]:
all_mild_ccevityes_scenarios_FRANKEN_df = unified_funcs.read_all_scenarios("../scenarios_inputs/franken/conditions_mild_harm_mild_good/cc_evitable_action_yes_stories.json", "../annotated_outputs/franken/conditions_mild_harm_mild_good/cc_evitable_action_yes_stories/")
# order the rows by SID and option
all_mild_ccevityes_scenarios_FRANKEN_df = all_mild_ccevityes_scenarios_FRANKEN_df.sort_values(by=["SID", "option"]).reset_index(drop=True)
display(HTML(all_mild_ccevityes_scenarios_FRANKEN_df.head().to_html()))
print("number of rows in all_mild_ccevityes_scenarios_FRANKEN_df:", len(all_mild_ccevityes_scenarios_FRANKEN_df))

,SID,option,event,being,C,I,K,utility
0,0,1: renovate the park.,The park closes temporarily,i,+,+,+,"{'residents of the community who use or value the park': '-55', 'i': '0'}"
1,0,1: renovate the park.,Residents lose access to the recreational space during the renovation,i,+,+,+,"{'residents of the community who use or value the park': '-68', 'i': '0'}"
2,0,1: renovate the park.,Residents experience inconvenience,i,+,-,+,"{'residents of the community who use or value the park': '-35', 'i': '0'}"
3,0,1: renovate the park.,Residents experience disappointment,i,+,-,+,"{'residents of the community who use or value the park': '-55', 'i': '-12'}"
4,0,1: renovate the park.,Park facilities are improved,i,+,+,+,"{'residents of the community who use or value the park': '84', 'i': '12'}"


number of rows in all_mild_ccevityes_scenarios_FRANKEN_df: 113


In [8]:
# if you really want all Franken scenarios, you need to loop over ["mild_harm_mild_good", "severe_harm_very_good"] WITH a nested loop over ["cc_evitable_action_yes_stories", "cc_evitable_action_no_stories"...] and sub them into the input json path and outputs dir path appropriately, and concatenate the resulting dataframes together

really_all_FRANKEN_df = []

intensities = ["mild_harm_mild_good", "severe_harm_very_good"]
causal_conditions = ["cc_evitable_action_yes_stories", "cc_evitable_prevention_no_stories", "cc_inevitable_action_yes_stories", "cc_inevitable_prevention_no_stories", "coc_evitable_action_yes_stories", "coc_evitable_prevention_no_stories", "coc_inevitable_action_yes_stories", "coc_inevitable_prevention_no_stories"]

for intensity in intensities:
    for causal_condition in causal_conditions:
        df = unified_funcs.read_all_scenarios(f"../scenarios_inputs/franken/conditions_{intensity}/{causal_condition}.json", f"../annotated_outputs/franken/conditions_{intensity}/{causal_condition}/")
        really_all_FRANKEN_df.append(df)
        # SPECIAL - add the intensity and causal_condition as columns to the dataframe
        really_all_FRANKEN_df[-1]["intensity"] = intensity
        really_all_FRANKEN_df[-1]["causal_condition"] = causal_condition
really_all_FRANKEN_df = pd.concat(really_all_FRANKEN_df, ignore_index=True)
# sort the intensity column by the order of the intensities in the intensities list above
really_all_FRANKEN_df["intensity"] = pd.Categorical(really_all_FRANKEN_df["intensity"], categories=intensities, ordered=True)
# sort them by the order of the causal conditions in the causal_conditions list above
really_all_FRANKEN_df["causal_condition"] = pd.Categorical(really_all_FRANKEN_df["causal_condition"], categories=causal_conditions, ordered=True)
really_all_FRANKEN_df = really_all_FRANKEN_df.sort_values(by=["intensity","causal_condition", "SID", "option"]).reset_index(drop=True)
# drop all the rows where SID is greater than 9
really_all_FRANKEN_df = really_all_FRANKEN_df[really_all_FRANKEN_df["SID"] <= 9]
display(HTML(really_all_FRANKEN_df[0:20].to_html()))
print("number of rows in really_all_FRANKEN_df:", len(really_all_FRANKEN_df))

,SID,option,event,being,C,I,K,utility,intensity,causal_condition
0,0,1: renovate the park.,The park closes temporarily,i,+,+,+,"{'residents of the community who use or value the park': '-55', 'i': '0'}",mild_harm_mild_good,cc_evitable_action_yes_stories
1,0,1: renovate the park.,Residents lose access to the recreational space during the renovation,i,+,+,+,"{'residents of the community who use or value the park': '-68', 'i': '0'}",mild_harm_mild_good,cc_evitable_action_yes_stories
2,0,1: renovate the park.,Residents experience inconvenience,i,+,-,+,"{'residents of the community who use or value the park': '-35', 'i': '0'}",mild_harm_mild_good,cc_evitable_action_yes_stories
3,0,1: renovate the park.,Residents experience disappointment,i,+,-,+,"{'residents of the community who use or value the park': '-55', 'i': '-12'}",mild_harm_mild_good,cc_evitable_action_yes_stories
4,0,1: renovate the park.,Park facilities are improved,i,+,+,+,"{'residents of the community who use or value the park': '84', 'i': '12'}",mild_harm_mild_good,cc_evitable_action_yes_stories
5,1,1: upgrade the server system.,I take my company's server system offline for a few hours,i,+,+,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories
6,1,1: upgrade the server system.,The company's operations temporarily halt,i,+,-,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories
7,1,1: upgrade the server system.,Data processing speed slightly improves after the upgrade,i,+,+,+,{'i': '12'},mild_harm_mild_good,cc_evitable_action_yes_stories
8,1,1: upgrade the server system.,Customers or users are unable to access server-dependent services during the downtime,i,+,-,+,{'i': '0'},mild_harm_mild_good,cc_evitable_action_yes_stories
9,2,1: implement the new teaching method.,Students attend class one hour earlier,i,+,+,+,"{'students required to attend class one hour earlier': '-15', 'students not required to attend class one hour earlier': '0', 'i': '0'}",mild_harm_mild_good,cc_evitable_action_yes_stories


number of rows in really_all_FRANKEN_df: 821


## Testing out `utility_df_maker` -- get average utility for each being across all events in a scenario

In [9]:
nie_utility_df = unified_funcs.utility_df_maker(really_all_NIE_df)
display(HTML(nie_utility_df[0:10].to_html()))
print("number of rows in nie_utility_df:", len(nie_utility_df))

,SID,option,being,average_utility
0,0,1: redirect the train to the side track,10 bus passengers on the current track,95.000000
1,0,1: redirect the train to the side track,2 bus passengers on the side track,-80.000000
2,0,1: redirect the train to the side track,i,-20.000000
3,0,2: keep the train on the same track,10 bus passengers on the current track,-75.000000
4,0,2: keep the train on the same track,2 bus passengers on the side track,25.000000
5,0,2: keep the train on the same track,i,-6.250000
6,1,1: push the cars,10 bus passengers on the bus currently on the track,63.888889
7,1,1: push the cars,train driver,0.000000
8,1,1: push the cars,several car drivers/passengers who would be pushed by my truck,-43.888889
9,1,1: push the cars,i,-24.444444


number of rows in nie_utility_df: 482


In [10]:
cheung_utility_df = unified_funcs.utility_df_maker(really_all_CHEUNG_df)
display(HTML(cheung_utility_df[0:10].to_html()))
print("number of rows in cheung_utility_df:", len(cheung_utility_df))

,SID,option,being,average_utility,scenario_title
0,1,1: kill the bird with the shovel,the injured bird,-30.00,Bird
1,1,1: kill the bird with the shovel,i,-36.00,Bird
2,2,1: kill the bird with the painless poison,the injured bird,-6.25,Bird
3,2,1: kill the bird with the painless poison,i,-18.00,Bird
4,3,1: go inside and do nothing,1 animal care facility worker,9.00,Bird
5,3,1: go inside and do nothing,1 bird,13.00,Bird
6,3,1: go inside and do nothing,i,-1.00,Bird
7,4,1: kill the bird with the shovel,the injured bird,-6.25,Bird
8,4,1: kill the bird with the shovel,i,-15.00,Bird
9,5,1: kill the bird with the painless poison,the injured bird,-5.00,Bird


number of rows in cheung_utility_df: 505


In [11]:
franken_utility_df = unified_funcs.utility_df_maker(really_all_FRANKEN_df)
display(HTML(franken_utility_df[0:10].to_html()))
print("number of rows in franken_utility_df:", len(franken_utility_df))

,SID,option,being,average_utility,intensity,causal_condition
0,0,1: renovate the park.,residents of the community who use or value the park,-25.800000,mild_harm_mild_good,cc_evitable_action_yes_stories
1,0,1: renovate the park.,i,0.000000,mild_harm_mild_good,cc_evitable_action_yes_stories
2,1,1: upgrade the server system.,i,-14.500000,mild_harm_mild_good,cc_evitable_action_yes_stories
3,2,1: implement the new teaching method.,students required to attend class one hour earlier,-23.333333,mild_harm_mild_good,cc_evitable_action_yes_stories
4,2,1: implement the new teaching method.,students not required to attend class one hour earlier,3.666667,mild_harm_mild_good,cc_evitable_action_yes_stories
5,2,1: implement the new teaching method.,i,24.000000,mild_harm_mild_good,cc_evitable_action_yes_stories
6,3,1: implement the new scheduling system.,patients needing to schedule their visits,-35.000000,mild_harm_mild_good,cc_evitable_action_yes_stories
7,3,1: implement the new scheduling system.,i,15.750000,mild_harm_mild_good,cc_evitable_action_yes_stories
8,4,1: change the menu.,regular customers,-18.200000,mild_harm_mild_good,cc_evitable_action_yes_stories
9,4,1: change the menu.,a few new customers,9.200000,mild_harm_mild_good,cc_evitable_action_yes_stories


number of rows in franken_utility_df: 481
